Este Notebook consolida todo o seu pipeline: desde o download do CIFAR-10 até a geração da tabela de erro comparativa. Ele integra a lógica de extração de dados, cálculo de covariância funcional ($\Sigma$) e a decomposição CP-ALS-Sigma.

---

# 🚀 Pipeline Completo: Reprodução da Tabela 1 (ResNet-18 @ CIFAR-10)

Este notebook executa os seguintes passos:

1. **Setup**: Prepara pastas e baixa o CIFAR-10 + ResNet-18.
2. **Covariância**: Calcula a matriz de covariância funcional para todas as camadas.
3. **Extração**: Separa os pesos e os fatores de Cholesky ($\Sigma^{1/2}$) por camada.
4. **Decomposição**: Executa o CP-ALS-Sigma usando o solver iterativo MINRES.
5. **Relatório**: Gera a tabela final de erro de reconstrução.

In [1]:
# =============================================================================
# 0. DEPENDÊNCIAS E CONFIGURAÇÕES
# =============================================================================
import torch
import torchvision
import os
import math
import random
import numpy as np
import cupy as cp
from tqdm import tqdm
from PIL import Image
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models.feature_extraction import create_feature_extractor, get_graph_node_names
from torchinfo import summary
from torch.utils.dlpack import from_dlpack, to_dlpack
from cupyx.scipy.sparse.linalg import minres, LinearOperator
from pathlib import Path

# Configurações Globais
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE_DIR = os.getcwd()
SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# Pastas necessárias
for folder in ['cifar10_data/images', 'outputs', 'weights']:
    os.makedirs(folder, exist_ok=True)

## 1. Setup de Dados e Modelo

Baixamos o modelo pré-treinado e preparamos as imagens de calibração do CIFAR-10.

In [2]:
# --- Download CIFAR-10 e Salvamento de Imagens ---
print("Baixando CIFAR-10...")
dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
num_images = len(dataset)

with open('dataset_files.txt', 'w') as f_files, open('dataset_labels.txt', 'w') as f_labels:
    for i in tqdm(range(num_images), desc="Salvando imagens"):
        img, label = dataset[i]
        img_name = f'img_{i}.png'
        img_path = os.path.join('cifar10_data/images', img_name)
        img.save(img_path)
        f_files.write(f'{img_name}\n')
        f_labels.write(f'{label}\n')

# --- Modelo ResNet-18 ---
print("Carregando ResNet-18...")
model_full = torchvision.models.resnet18(weights="IMAGENET1K_V1")
torch.save(model_full, 'weights/resnet18_full.pt')

# --- Definição da Transformação ---
from torchvision import transforms
def get_transform():
    return transforms.Compose([
        transforms.Resize((32, 32)),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
my_transform = get_transform()

Baixando CIFAR-10...


100%|██████████| 170M/170M [00:13<00:00, 12.5MB/s] 
Salvando imagens: 100%|██████████| 50000/50000 [00:06<00:00, 7382.55it/s]


Carregando ResNet-18...


## 2. Cálculo da Covariância ($\Sigma$)

Integração da lógica do seu script `Compute_Covariance.py` para gerar o dicionário de Sigmas.

In [3]:
def run_covariance_step():
    from Compute_Covariance import _main
    import argparse

    args = argparse.Namespace(
        model='weights/resnet18_full.pt',
        transf=None, # Passamos o objeto transform direto se necessário, ou salvamos
        root='cifar10_data/images',
        dataset_files='dataset_files.txt',
        dataset_labels='dataset_labels.txt',
        batch_size=50,
        batch_size_mean=None,
        batch_size_product=None,
        bias=False,
        output='outputs/sigma_resnet18',
        workers=4
    )

    # Salvamos o transform para o script ler (como o seu _main espera um path)
    torch.save(my_transform, 'my_transform.pt')
    args.transf = 'my_transform.pt'

    print("\nIniciando cálculo de Covariância Funcional...")
    _main(args)

run_covariance_step()


Iniciando cálculo de Covariância Funcional...
Started at: 2026-03-26 10:41:24
Batch size when computing E(X_i)E(Xj): 50
Batch size when computing E(X_i Xj): 50
Using GPU: NVIDIA GeForce RTX 4090
Using 4 workers.
Is CUDA available: True
Computing E(X_i)E(Xj)...
Computing the mean of the input of all the conv2d layers...
Start get_mean_pre_conv_input_full
{'x': 'conv1', 'maxpool': 'layer1.0.conv1', 'layer1.0.relu': 'layer1.0.conv2', 'layer1.0.relu_1': 'layer1.1.conv1', 'layer1.1.relu': 'layer1.1.conv2', 'layer1.1.relu_1': 'layer2.0.conv1', 'layer2.0.relu': 'layer2.0.conv2', 'layer2.0.relu_1': 'layer2.1.conv1', 'layer2.1.relu': 'layer2.1.conv2', 'layer2.1.relu_1': 'layer3.0.conv1', 'layer3.0.relu': 'layer3.0.conv2', 'layer3.0.relu_1': 'layer3.1.conv1', 'layer3.1.relu': 'layer3.1.conv2', 'layer3.1.relu_1': 'layer4.0.conv1', 'layer4.0.relu': 'layer4.0.conv2', 'layer4.0.relu_1': 'layer4.1.conv1', 'layer4.1.relu': 'layer4.1.conv2'}
{'layer1.1.conv2', 'layer3.0.conv2', 'layer3.1.conv2', 'laye

100%|██████████| 1000/1000 [00:02<00:00, 456.19it/s]


Full mean RAM use: 1.176899584 GB RAM
GPU memory: 2.58e+01 GB VRAM GPU memory allocated: 2.47e+00 GB VRAM GPU memory free: 2.33e+01 GB VRAM
Full mean are in cuda:0
Computing the product of the mean of the input of all the conv2d layers...
Computing the product of the mean of the input of layer1.1.conv2...
Computing the product of the mean of the input of layer3.0.conv2...
Computing the product of the mean of the input of layer3.1.conv2...
Computing the product of the mean of the input of layer2.1.conv1...
Computing the product of the mean of the input of layer2.1.conv2...
Computing the product of the mean of the input of layer4.1.conv1...
Computing the product of the mean of the input of layer1.0.conv2...
Computing the product of the mean of the input of conv1...
Computing the product of the mean of the input of layer3.1.conv1...
Computing the product of the mean of the input of layer1.1.conv1...
Computing the product of the mean of the input of layer4.0.conv2...
Computing the product 

100%|██████████| 1000/1000 [00:31<00:00, 31.37it/s]


Full product RAM use: 1.340604416 GB RAM
GPU memory: 2.58e+01 GB VRAM GPU memory allocated: 2.07e+01 GB VRAM GPU memory free: 5.05e+00 GB VRAM
Full product is in cuda:0
Reshaping the mean of the product of the input of all the conv2d layers...
Mean product is in cuda:0
Computing covariance...
Time to compute the covariance: 34.36s
Saving the covariance...
Computing the Cholesky decomposition of the covariances...
The tensor of layer layer3.1.conv2 is not positive definite.
Added 1e-09 to the diagonal of the tensor of layer layer3.1.conv2.
The tensor of layer layer4.1.conv1 is not positive definite.
Added 1e-09 to the diagonal of the tensor of layer layer4.1.conv1.
The tensor of layer layer4.0.conv2 is not positive definite.
Added 1e-09 to the diagonal of the tensor of layer layer4.0.conv2.
The tensor of layer layer4.1.conv2 is not positive definite.
Added 1e-09 to the diagonal of the tensor of layer layer4.1.conv2.
The tensor of layer layer4.0.conv1 is not positive definite.
Added 1e-0

## 3. Extração de Pesos e Sigmas por Camada

Preparamos os arquivos individuais para as camadas da Tabela 1.

In [6]:
LAYERS_MAP = {
    'layer1.0.conv2': 134,
    'layer2.0.conv1': 140,
    'layer3.0.conv1': 291,
    'layer3.1.conv1': 520,
    'layer4.1.conv1': 1023,
    'layer4.1.conv2': 1167
}

def extract_layers():
    print("\nExtraindo pesos e fatores de Cholesky...")
    sigmas_dict = torch.load('outputs/sigma_resnet18_cholesky_covariance.pt', weights_only=False)
    model = torch.load('weights/resnet18_full.pt', weights_only=False)
    modules = dict(model.named_modules())

    for layer_name in LAYERS_MAP.keys():
        # Salvar Sigma
        torch.save(sigmas_dict[layer_name], f'outputs/sigma_resnet18_{layer_name}.pt')

        # Salvar Pesos formatados [Out, In, H*W]
        target = modules[layer_name]
        w_raw = target.weight.data
        out_c, in_c, h, w = w_raw.shape
        w_fmt = w_raw.reshape(out_c, in_c, h*w).contiguous()
        torch.save(w_fmt, f'weights/{layer_name.replace(".", "_")}.pt')
        print(f"Camada {layer_name} extraída.")

extract_layers()


Extraindo pesos e fatores de Cholesky...
Camada layer1.0.conv2 extraída.
Camada layer2.0.conv1 extraída.
Camada layer3.0.conv1 extraída.
Camada layer3.1.conv1 extraída.
Camada layer4.1.conv1 extraída.
Camada layer4.1.conv2 extraída.


## 4. Decomposição CP-ALS-Sigma

Implementação do algoritmo usando os fatores extraídos e o solver **MINRES**.

In [7]:
# Funções utilitárias DLPack
def torch_to_cupy(t): return cp.from_dlpack(t)
def cupy_to_torch(c): return from_dlpack(c)

def run_all_decompositions():
    results = {}
    from CP_ALS_Sigma import cp_als_sigma

    print("\nIniciando Decomposições CP-ALS-Sigma...")
    for layer_name, rank in LAYERS_MAP.items():
        print(f"\n>>> Processando {layer_name} (Rank {rank})")

        W = torch.load(f'weights/{layer_name.replace(".", "_")}.pt', weights_only=False)
        S_half = torch.load(f'outputs/sigma_resnet18_{layer_name}.pt', weights_only=False)

        factors = cp_als_sigma(W, rank, S_half, n_iter_max=100, verbose=0)

        save_path = f"outputs/cp_factors_{layer_name.replace('.', '_')}_rank{rank}.pt"
        torch.save(factors, save_path)

        # Guardamos os caminhos para o cálculo final
        results[layer_name] = {
            'weights': f'weights/{layer_name.replace(".", "_")}.pt',
            'factors': save_path,
            'sigma': f'outputs/sigma_resnet18_{layer_name}.pt',
            'rank': rank
        }
    return results

layer_files = run_all_decompositions()


Iniciando Decomposições CP-ALS-Sigma...

>>> Processando layer1.0.conv2 (Rank 134)

>>> Processando layer2.0.conv1 (Rank 140)

>>> Processando layer3.0.conv1 (Rank 291)

>>> Processando layer3.1.conv1 (Rank 520)

>>> Processando layer4.1.conv1 (Rank 1023)

>>> Processando layer4.1.conv2 (Rank 1167)


## 5. Tabela de Resultados (Reprodução da Tabela 1)

Cálculo final do erro relativo na norma funcional $\Sigma$.

In [8]:
def calculate_table():
    print(f"\n{'Layer':<15} | {'Rank':<5} | {'Sigma Error':<12}")
    print("-" * 40)

    # Valores de referência do seu prompt (Tabela 1 original)
    ref_table = {
        'layer1.0.conv2': 0.053, 'layer2.0.conv1': 0.118,
        'layer3.0.conv1': 0.094, 'layer3.1.conv1': 0.070,
        'layer4.1.conv1': 0.063, 'layer4.1.conv2': 0.051
    }

    for name, info in layer_files.items():
        K = torch.load(info['weights'], weights_only=False).to(DEVICE)
        factors = torch.load(info['factors'], weights_only=False)
        S_half = torch.load(info['sigma'], weights_only=False).to(DEVICE)

        # Reconstrução
        u_t, u_s, u_h, u_w = [f.to(DEVICE) for f in factors]
        K_hat = torch.einsum('tr,sr,hr,wr->tshw', u_t, u_s, u_h, u_w)

        # Unfolding e Erro
        out_c = K.shape[0]
        K_flat = K.reshape(out_c, -1)
        K_hat_flat = K_hat.reshape(out_c, -1)

        num = torch.norm(torch.matmul(K_flat - K_hat_flat, S_half))
        den = torch.norm(torch.matmul(K_flat, S_half))

        error = (num / den).item()

        print(f"{name:<15} | {info['rank']:<5} | {error:<12.3f} (Ref: {ref_table[name]})")

calculate_table()


Layer           | Rank  | Sigma Error 
----------------------------------------
layer1.0.conv2  | 134   | 0.087        (Ref: 0.053)
layer2.0.conv1  | 140   | 0.137        (Ref: 0.118)
layer3.0.conv1  | 291   | 0.125        (Ref: 0.094)
layer3.1.conv1  | 520   | 0.140        (Ref: 0.07)
layer4.1.conv1  | 1023  | 0.366        (Ref: 0.063)
layer4.1.conv2  | 1167  | 0.331        (Ref: 0.051)


### Observações Técnicas Importantes:

* **Norma Funcional**: O notebook utiliza a definição $\|(\mathcal{K} - \hat{\mathcal{K}})_{(1)} \Sigma^{1/2}\|_F$, que é a métrica padrão para compressão baseada em dados.
* **Solver MINRES**: A decomposição utiliza o solver iterativo para o Modo-0 (canais de saída), garantindo estabilidade numérica em sistemas mal-condicionados causados pela matriz $\Sigma$.
* **DLPack**: A transferência de dados entre **PyTorch** e **CuPy** é feita via `DLPack`, o que evita cópias desnecessárias de memória na GPU e acelera o processo de otimização.

Seria útil para você adicionar uma célula final que **substitua automaticamente** as camadas originais do ResNet-18 pelos fatores comprimidos para testar a acurácia (Top-1) no CIFAR-10?